# Stage 4 · YOLO26s controlled long training

Review the saved run on the frozen RoadGuardAI snapshot. The training budget was 50 epochs with validation patience of 10. The figures and tables come from the experiment archive.

In [ ]:
import json

import pandas as pd
from IPython.display import Image, display

from roadguard.data.dataset import CLASS_NAMES, RDD_CLASSES
from roadguard.training.archive import open_experiment
from roadguard.training.stage4_report import MAP
from roadguard.utils.paths import project_root

root = project_root()
result = json.loads((root / "artifacts/experiments/stage4_result.json").read_text(encoding="utf-8"))
run = root / result["run"]
archive = open_experiment(run)
metadata = archive.metadata
preflight = json.loads((run / "preflight.json").read_text(encoding="utf-8"))
print(run.name)

## CUDA and frozen dataset

Preflight recorded the CUDA operation and TorchVision NMS check on the training device. The run descriptor contains the frozen train and validation partitions.

In [ ]:
display(pd.Series(preflight["environment"], name="Recorded environment").to_frame())
print("CUDA and TorchVision NMS:", preflight["cuda_torchvision_nms"])
print("Dataset fingerprint:", metadata["dataset_fingerprint"])
splits = ("train", "val")
display(pd.DataFrame({split: metadata["dataset_summary"][split] for split in splits}).T)
display(
    pd.DataFrame(
        {split: metadata["dataset_summary"][split]["annotations_per_class"] for split in splits}
    ).rename_axis("Class")
)
print((run / "dataset.yaml").read_text(encoding="utf-8"))

## Model, configuration and training launch

The run initialized YOLO26s from the validation-best checkpoint selected in Stage 3B and created a fresh AdamW optimizer. The launch command from the repository root was `python scripts/train_stage4.py`. It delegates fitting to the shared `scripts/train_detector.py` pipeline.

In [ ]:
display(pd.Series(metadata["config"]["model"], name="Model settings").to_frame())
training = dict(metadata["config"]["training"])
augmentation = training.pop("augmentation")
display(pd.Series(training, name="Training settings").to_frame())
display(pd.Series(augmentation, name="Augmentation").to_frame())
print("Initialization SHA-256:", preflight["initialization"]["sha256"])

## Epoch history and learning curves

The CSV records training and validation losses, precision, recall, mAP50 and mAP50–95 for every completed epoch.

In [ ]:
history = pd.DataFrame(archive.epoch_metrics).apply(pd.to_numeric).set_index("epoch")
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(history.round(5))
display(Image(filename=str(run / "results.png"), width=1050))

## Validation metrics and selected checkpoint

The best checkpoint was selected using validation mAP50–95. The table reports a separate validation pass on the saved checkpoint.

In [ ]:
metrics = result["validation_metrics"]
comparison = pd.DataFrame(
    {"Stage 3B": result["stage3b_metrics"], "Stage 4": metrics},
    index=["precision", "recall", "map50", "map50_95"],
)
comparison["Difference"] = comparison["Stage 4"] - comparison["Stage 3B"]
display(comparison.round(4))
per_class = pd.DataFrame(
    {code: metrics["per_class"][CLASS_NAMES[index]] for code, index in RDD_CLASSES.items()}
).T
display(per_class.round(4))
display(pd.Series(archive.best, name="Best checkpoint").to_frame())

## Confusion matrix and threshold curves

The normalized confusion matrix shows where the detector misses or confuses classes. PR and F1 show how the validation set responds as confidence changes.

In [ ]:
for name in ("confusion_matrix_normalized.png", "BoxPR_curve.png", "BoxF1_curve.png"):
    display(Image(filename=str(run / name), width=880))

## Validation predictions and error analysis

These validation examples use the archived confidence and IoU thresholds. The counts below describe the sample shown by the review pipeline.

In [ ]:
review = archive.validation_review
print("Validation examples:", review["sample_count"])
print("Confidence:", review["confidence_threshold"], "IoU:", review["match_iou_threshold"])
display(pd.DataFrame(review["per_class"]).T.fillna(0).astype(int))
print("Sample totals:", review["totals"])
print("Class confusions:", review["class_confusions"])
for sample in review["samples"][:6]:
    display(Image(filename=str(run / sample["figure"]), width=900))

## Training outcome

Compare the final two validation windows and the number of epochs since the last improvement. The Arabic report records the final decision and the evidence for it.

In [ ]:
trend = result["trend"]
display(pd.Series(result["stopping"], name="Stopping state").to_frame())
display(pd.Series(trend["mean_change"], name="Final-window mean change").to_frame())
print(f"Best validation mAP50–95 gain: {result['absolute_map50_95_gain']:+.4f}")
print(f"Final epoch validation mAP50–95: {history[MAP].iloc[-1]:.4f}")
print("New best in final window:", trend["new_best_in_final_window"])
print("Possible overfitting signal:", trend["overfitting_signal"])
print("Extension worth considering:", trend["extension_worth_considering"])
print("Selected candidate:", result["selected_checkpoint"])
print("Frozen train/validation checks passed:", result["integrity"]["train_val_checksums_match"])
print("Test evaluation performed:", result["test_evaluated"])